# Red Cedar — Training Settings

This notebook shows training settings you can change, one section per setting. Each section trains a small model on the same dataset (4 qubits, 1 layer) and tests it, so the effect of the setting is easy to see.

Run the setup cell first; after that, each section can be run on its own. For the basic workflow see `01_quick_start.ipynb`. All settings are listed in the settings table of the redcedar README.

In [1]:
import redcedar

# Connect to the server
# Reads REDCEDAR_URL and REDCEDAR_API_KEY from environment, or pass explicitly:
# client = redcedar.connect(url="http://127.0.0.1:8002", api_key="your-key")
# With the Red Cedar source installed (pip install -e), local mode calls it directly:
# client = redcedar.connect(local=True)
client = redcedar.connect()
print(f"Red Cedar SDK v{redcedar.__version__}")
status = client.status()
print(f"Server: {client.url or 'local mode, in this process'} (version {status['server_version'] or 'unknown'}, ok={status['ok']})")

dataset = "wdbc"

# Small, quick settings shared by every section; each section adds one setting.
base = {"num_qubits": 4, "min_layers": 1, "max_layers": 1}

def train_and_test(settings):
    """Train with these settings, test on the held-out data, return (train_id, accuracy, loss)."""
    result = client.training.run(dataset, settings=settings)
    test = client.training.evaluate(result["train_id"])
    return result["train_id"], test["overall_accuracy"], test["overall_loss"]

Red Cedar SDK v0.93.0
Server: http://127.0.0.1:8002 (version 0.92-18-g1706a0dc-dirty, ok=True)


## 1. `seed`: repeatable runs

Training starts from random parameters, so two runs with the same settings usually end differently. `seed` fixes the starting parameters (and, on a CUDA-Q server, the circuit sampling), so a run can be repeated exactly. Leave it out, or set it to `None`, for a random start.

On a model this small the test accuracy often comes out the same; the loss shows the difference between runs.

In [2]:
runs = {
    "seed 123, run 1": dict(base, seed=123),
    "seed 123, run 2": dict(base, seed=123),
    "no seed":         dict(base),
}
results = {name: train_and_test(settings) for name, settings in runs.items()}
for name, (train_id, accuracy, loss) in results.items():
    print(f"{name:16s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

same = results["seed 123, run 1"][1:] == results["seed 123, run 2"][1:]
print(f"\nSame seed, same result: {same}")

seed 123, run 1  accuracy=75.4%  loss=0.269245  model cf1f8478
seed 123, run 2  accuracy=75.4%  loss=0.269245  model f76c5384
no seed          accuracy=75.4%  loss=0.253591  model 2b74b9a2

Same seed, same result: True


## 2. `op_direction`: order of the operators

Each layer pairs the prediction qubit with every data qubit in turn. `op_direction` sets the order: `"backward"` (the default) starts at the highest data qubit, `"forward"` at the lowest. The number of parameters is the same either way; the results differ because the optimizer visits the parameters in a different order.

Both runs below use the same seed, so the direction is the only difference. Neither direction is better in general: across many seeds and datasets the results were close.

In [3]:
for direction in ["backward", "forward"]:
    train_id, accuracy, loss = train_and_test(dict(base, seed=123, op_direction=direction))
    print(f"{direction:8s} accuracy={accuracy:.1%}  loss={loss:.6f}  model {train_id[:8]}")

backward accuracy=75.4%  loss=0.269245  model 5a00aa86
forward  accuracy=75.4%  loss=0.267192  model 45d42825


## Settings are saved with the model

Each trained model keeps its settings, and the model list shows them, so any run can be traced back and repeated.

In [4]:
# The models trained on this dataset (the list has every model on the server)
models = [m for m in client.inference.models()["models"] if m["dataset_name"] == dataset]
for m in models[:5]:
    p = m["training_params"]
    print(f"{m['train_id'][:8]}  {m['dataset']:12s} seed={p['seed']}  op_direction={p['op_direction']}")

2b74b9a2  WDBC (Breast Cancer) seed=None  op_direction=backward
45d42825  WDBC (Breast Cancer) seed=123  op_direction=forward
5a00aa86  WDBC (Breast Cancer) seed=123  op_direction=backward
cf1f8478  WDBC (Breast Cancer) seed=123  op_direction=backward
f76c5384  WDBC (Breast Cancer) seed=123  op_direction=backward
